In [ ]:
# %% [markdown]
# # Notebook 02: Exploratory Data Analysis & Statistical Analysis
# ## VeritasFinancial - Banking Fraud Detection System
# 
# **Objective**: Perform comprehensive exploratory data analysis to uncover patterns, relationships, and insights in banking transaction data
# 
# **Key Tasks**:
# 1. Univariate analysis of all features
# 2. Bivariate analysis with target variable
# 3. Multivariate analysis and correlations
# 4. Temporal pattern analysis
# 5. Behavioral pattern extraction
# 6. Feature importance ranking
# 7. Visualization dashboard creation

# %% [markdown]
# ### 1. Import Required Libraries

# %%
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical libraries
from scipy import stats
from scipy.stats import norm, skew, kurtosis
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Machine learning libraries for feature selection
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Utilities
import warnings
from datetime import datetime, timedelta
import logging
import json
import os

# Suppress warnings
warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

# %% [markdown]
# ### 2. Load Processed Data

# %%
def load_processed_data(data_path='../data/processed/transactions_processed.parquet'):
    """
    Load processed data from previous notebook.
    
    Args:
        data_path (str): Path to processed data
        
    Returns:
        pd.DataFrame: Loaded dataframe
    """
    try:
        df = pd.read_parquet(data_path)
        logger.info(f"Data loaded successfully from {data_path}")
        print(f"✅ Data loaded. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.warning(f"Processed data not found. Loading raw data instead...")
        # Fall back to raw data
        df = pd.read_csv('../data/raw/creditcard.csv')
        logger.info(f"Raw data loaded. Shape: {df.shape}")
        return df

# Load data
df = load_processed_data()

# Display basic info
print("\n" + "="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Total transactions: {len(df):,}")
print(f"Total features: {df.shape[1]}")
print(f"Fraud cases: {df['Class'].sum():,} ({df['Class'].mean()*100:.4f}%)")
print(f"Non-fraud cases: {(df['Class'] == 0).sum():,} ({(1-df['Class'].mean())*100:.4f}%)")

# %% [markdown]
# ### 3. Univariate Analysis - Distribution Analysis

# %%
class UnivariateAnalyzer:
    """
    Comprehensive univariate analysis for all features.
    """
    
    def __init__(self, df):
        self.df = df
        self.results = {}
        
    def analyze_all_features(self):
        """
        Perform univariate analysis on all features.
        """
        print("\n" + "="*60)
        print("UNIVARIATE ANALYSIS")
        print("="*60)
        
        # Separate numerical and categorical features
        self.numerical_cols = self.df.select_dtypes(include=[np.number]).columns
        self.numerical_cols = [col for col in self.numerical_cols if col != 'Class']
        
        # Analyze each numerical feature
        for col in self.numerical_cols:
            self._analyze_numerical_feature(col)
        
        # Create summary visualizations
        self._plot_distribution_grid()
        self._plot_qq_plots()
        self._plot_skewness_kurtosis()
        
        return self.results
    
    def _analyze_numerical_feature(self, col):
        """
        Analyze a single numerical feature.
        
        Args:
            col (str): Column name
        """
        # Basic statistics
        stats_dict = {
            'mean': self.df[col].mean(),
            'median': self.df[col].median(),
            'mode': self.df[col].mode().iloc[0] if not self.df[col].mode().empty else np.nan,
            'std': self.df[col].std(),
            'var': self.df[col].var(),
            'min': self.df[col].min(),
            'max': self.df[col].max(),
            'range': self.df[col].max() - self.df[col].min(),
            'q1': self.df[col].quantile(0.25),
            'q3': self.df[col].quantile(0.75),
            'iqr': self.df[col].quantile(0.75) - self.df[col].quantile(0.25),
            'skewness': skew(self.df[col].dropna()),
            'kurtosis': kurtosis(self.df[col].dropna())
        }
        
        # Normality tests
        if len(self.df[col].dropna()) > 3:
            # Shapiro-Wilk test (for normality)
            if len(self.df[col].dropna()) <= 5000:
                shapiro_stat, shapiro_p = stats.shapiro(self.df[col].dropna().sample(min(5000, len(self.df[col]))))
                stats_dict['shapiro_stat'] = shapiro_stat
                stats_dict['shapiro_p'] = shapiro_p
                stats_dict['is_normal'] = shapiro_p > 0.05
            
            # Kolmogorov-Smirnov test
            ks_stat, ks_p = stats.kstest(self.df[col].dropna(), 'norm', 
                                         args=(self.df[col].mean(), self.df[col].std()))
            stats_dict['ks_stat'] = ks_stat
            stats_dict['ks_p'] = ks_p
        
        # Outlier counts using IQR
        Q1 = stats_dict['q1']
        Q3 = stats_dict['q3']
        IQR = stats_dict['iqr']
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = self.df[(self.df[col] < lower_bound) | (self.df[col] > upper_bound)]
        stats_dict['outlier_count'] = len(outliers)
        stats_dict['outlier_percent'] = (len(outliers) / len(self.df)) * 100
        
        self.results[col] = stats_dict
    
    def _plot_distribution_grid(self):
        """
        Create a grid of distribution plots for top features.
        """
        # Select top features based on variance or importance
        feature_variance = {col: self.results[col]['var'] for col in self.numerical_cols}
        top_features = sorted(feature_variance, key=feature_variance.get, reverse=True)[:16]
        
        # Create subplot grid
        n_cols = 4
        n_rows = (len(top_features) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
        axes = axes.flatten()
        
        for idx, feature in enumerate(top_features):
            ax = axes[idx]
            
            # Plot histogram with KDE
            ax.hist(self.df[feature], bins=50, density=True, alpha=0.7, 
                   color='skyblue', edgecolor='black')
            self.df[feature].plot.kde(ax=ax, color='red', linewidth=2)
            
            # Add statistics
            mean_val = self.results[feature]['mean']
            median_val = self.results[feature]['median']
            ax.axvline(mean_val, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
            ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.3f}')
            
            ax.set_title(f'{feature}\nSkew: {self.results[feature]["skewness"]:.3f}, Kurt: {self.results[feature]["kurtosis"]:.3f}')
            ax.set_xlabel('')
            ax.set_ylabel('Density')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        
        # Hide empty subplots
        for idx in range(len(top_features), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('Distribution of Top 16 Features', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    
    def _plot_qq_plots(self):
        """
        Create Q-Q plots for normality assessment.
        """
        # Select features for Q-Q plots
        selected_features = self.numerical_cols[:9]  # First 9 features
        
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()
        
        for idx, feature in enumerate(selected_features):
            ax = axes[idx]
            
            # Generate Q-Q plot
            stats.probplot(self.df[feature].dropna(), dist="norm", plot=ax)
            ax.set_title(f'Q-Q Plot: {feature}')
            ax.grid(True, alpha=0.3)
            
            # Add normality test result
            if 'shapiro_p' in self.results[feature]:
                p_value = self.results[feature]['shapiro_p']
                normality = "Normal" if p_value > 0.05 else "Non-normal"
                ax.text(0.05, 0.95, f'{normality} (p={p_value:.4f})', 
                       transform=ax.transAxes, fontsize=10,
                       verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle('Q-Q Plots for Normality Assessment', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    
    def _plot_skewness_kurtosis(self):
        """
        Visualize skewness and kurtosis of features.
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Skewness plot
        skewness_values = [self.results[col]['skewness'] for col in self.numerical_cols]
        skewness_df = pd.DataFrame({'Feature': self.numerical_cols, 'Skewness': skewness_values})
        skewness_df = skewness_df.sort_values('Skewness', ascending=False)
        
        colors = ['red' if abs(x) > 1 else 'orange' if abs(x) > 0.5 else 'green' 
                 for x in skewness_df['Skewness'].head(20)]
        
        ax1.barh(range(20), skewness_df['Skewness'].head(20), color=colors)
        ax1.set_yticks(range(20))
        ax1.set_yticklabels(skewness_df['Feature'].head(20))
        ax1.set_xlabel('Skewness')
        ax1.set_title('Top 20 Features by Absolute Skewness', fontweight='bold')
        ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        ax1.axvline(x=0.5, color='orange', linestyle='--', alpha=0.5)
        ax1.axvline(x=-0.5, color='orange', linestyle='--', alpha=0.5)
        ax1.axvline(x=1, color='red', linestyle='--', alpha=0.5)
        ax1.axvline(x=-1, color='red', linestyle='--', alpha=0.5)
        ax1.grid(True, alpha=0.3, axis='x')
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='green', label='|Skew| < 0.5 (Symmetric)'),
            Patch(facecolor='orange', label='0.5 ≤ |Skew| < 1 (Moderate)'),
            Patch(facecolor='red', label='|Skew| ≥ 1 (Highly Skewed)')
        ]
        ax1.legend(handles=legend_elements, loc='lower right')
        
        # Kurtosis plot
        kurtosis_values = [self.results[col]['kurtosis'] for col in self.numerical_cols]
        kurtosis_df = pd.DataFrame({'Feature': self.numerical_cols, 'Kurtosis': kurtosis_values})
        kurtosis_df = kurtosis_df.sort_values('Kurtosis', ascending=False)
        
        colors = ['red' if x > 3 else 'orange' if x > 0 else 'green' 
                 for x in kurtosis_df['Kurtosis'].head(20)]
        
        ax2.barh(range(20), kurtosis_df['Kurtosis'].head(20), color=colors)
        ax2.set_yticks(range(20))
        ax2.set_yticklabels(kurtosis_df['Feature'].head(20))
        ax2.set_xlabel('Kurtosis')
        ax2.set_title('Top 20 Features by Kurtosis', fontweight='bold')
        ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        ax2.axvline(x=3, color='red', linestyle='--', alpha=0.5)
        ax2.grid(True, alpha=0.3, axis='x')
        
        # Add legend
        legend_elements = [
            Patch(facecolor='green', label='Kurtosis < 0 (Platykurtic)'),
            Patch(facecolor='orange', label='0 ≤ Kurtosis < 3 (Mesokurtic)'),
            Patch(facecolor='red', label='Kurtosis ≥ 3 (Leptokurtic)')
        ]
        ax2.legend(handles=legend_elements, loc='lower right')
        
        plt.suptitle('Skewness and Kurtosis Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Perform univariate analysis
univariate_analyzer = UnivariateAnalyzer(df)
univariate_results = univariate_analyzer.analyze_all_features()

# %% [markdown]
# ### 4. Bivariate Analysis - Feature vs Target

# %%
class BivariateAnalyzer:
    """
    Bivariate analysis between features and target variable.
    """
    
    def __init__(self, df, target_col='Class'):
        self.df = df
        self.target_col = target_col
        self.results = {}
        
    def analyze_all_features(self):
        """
        Analyze relationship between each feature and target.
        """
        print("\n" + "="*60)
        print("BIVARIATE ANALYSIS (Feature vs Target)")
        print("="*60)
        
        # Separate data by class
        self.fraud_df = self.df[self.df[self.target_col] == 1]
        self.non_fraud_df = self.df[self.df[self.target_col] == 0]
        
        # Get numerical features
        numerical_cols = self.df.select_dtypes(include=[np.number]).columns
        numerical_cols = [col for col in numerical_cols if col != self.target_col]
        
        # Analyze each feature
        for col in numerical_cols:
            self._analyze_feature_vs_target(col)
        
        # Create visualizations
        self._plot_feature_comparisons()
        self._plot_conditional_distributions()
        self._plot_statistical_tests()
        
        return self.results
    
    def _analyze_feature_vs_target(self, col):
        """
        Analyze relationship between a feature and target.
        
        Args:
            col (str): Feature name
        """
        # Basic statistics by class
        fraud_mean = self.fraud_df[col].mean()
        fraud_std = self.fraud_df[col].std()
        fraud_median = self.fraud_df[col].median()
        
        non_fraud_mean = self.non_fraud_df[col].mean()
        non_fraud_std = self.non_fraud_df[col].std()
        non_fraud_median = self.non_fraud_df[col].median()
        
        # Mean difference
        mean_diff = abs(fraud_mean - non_fraud_mean)
        mean_diff_ratio = mean_diff / (non_fraud_std + 1e-10)
        
        # Statistical tests
        # T-test
        t_stat, t_p = stats.ttest_ind(self.fraud_df[col], self.non_fraud_df[col], equal_var=False)
        
        # Mann-Whitney U test (non-parametric)
        u_stat, u_p = stats.mannwhitneyu(self.fraud_df[col], self.non_fraud_df[col])
        
        # Kolmogorov-Smirnov test (distribution difference)
        ks_stat, ks_p = stats.ks_2samp(self.fraud_df[col], self.non_fraud_df[col])
        
        # Effect size (Cohen's d)
        n1, n2 = len(self.fraud_df[col]), len(self.non_fraud_df[col])
        pooled_std = np.sqrt(((n1 - 1) * fraud_std**2 + (n2 - 1) * non_fraud_std**2) / (n1 + n2 - 2))
        cohens_d = abs(fraud_mean - non_fraud_mean) / pooled_std if pooled_std > 0 else 0
        
        # Information value (for feature importance)
        iv = self._calculate_information_value(col)
        
        # Store results
        self.results[col] = {
            'fraud_mean': fraud_mean,
            'fraud_std': fraud_std,
            'fraud_median': fraud_median,
            'non_fraud_mean': non_fraud_mean,
            'non_fraud_std': non_fraud_std,
            'non_fraud_median': non_fraud_median,
            'mean_diff': mean_diff,
            'mean_diff_ratio': mean_diff_ratio,
            't_stat': t_stat,
            't_p_value': t_p,
            'u_stat': u_stat,
            'u_p_value': u_p,
            'ks_stat': ks_stat,
            'ks_p_value': ks_p,
            'cohens_d': cohens_d,
            'information_value': iv,
            'effect_magnitude': self._interpret_effect_size(cohens_d),
            'significant': t_p < 0.05
        }
    
    def _calculate_information_value(self, col, n_bins=10):
        """
        Calculate Information Value (IV) for feature.
        
        Args:
            col (str): Feature name
            n_bins (int): Number of bins
            
        Returns:
            float: Information value
        """
        # Bin the feature
        binned = pd.cut(self.df[col], bins=n_bins)
        
        # Calculate WOE and IV
        iv_df = pd.crosstab(binned, self.df[self.target_col]).reset_index()
        iv_df.columns = ['bin', 'non_event', 'event']
        
        # Add small constant to avoid division by zero
        iv_df['non_event'] = iv_df['non_event'] + 0.5
        iv_df['event'] = iv_df['event'] + 0.5
        
        # Calculate percentages
        iv_df['non_event_pct'] = iv_df['non_event'] / iv_df['non_event'].sum()
        iv_df['event_pct'] = iv_df['event'] / iv_df['event'].sum()
        
        # Calculate WOE
        iv_df['woe'] = np.log(iv_df['non_event_pct'] / iv_df['event_pct'])
        
        # Calculate IV
        iv_df['iv'] = (iv_df['non_event_pct'] - iv_df['event_pct']) * iv_df['woe']
        
        return iv_df['iv'].sum()
    
    def _interpret_effect_size(self, d):
        """
        Interpret Cohen's d effect size.
        
        Args:
            d (float): Cohen's d
            
        Returns:
            str: Interpretation
        """
        if d < 0.2:
            return "Negligible"
        elif d < 0.5:
            return "Small"
        elif d < 0.8:
            return "Medium"
        else:
            return "Large"
    
    def _plot_feature_comparisons(self):
        """
        Create comparison plots for features.
        """
        # Sort features by effect size
        sorted_features = sorted(self.results.items(), key=lambda x: x[1]['cohens_d'], reverse=True)
        top_features = [f[0] for f in sorted_features[:12]]
        
        # Create subplot grid
        n_cols = 3
        n_rows = 4
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 20))
        axes = axes.flatten()
        
        for idx, feature in enumerate(top_features):
            ax = axes[idx]
            
            # Create boxplot
            box_data = [self.non_fraud_df[feature], self.fraud_df[feature]]
            bp = ax.boxplot(box_data, labels=['Non-Fraud', 'Fraud'], patch_artist=True)
            
            # Color boxes
            bp['boxes'][0].set_facecolor('lightblue')
            bp['boxes'][1].set_facecolor('lightcoral')
            
            # Add individual points for fraud (to show distribution)
            if len(self.fraud_df[feature]) < 1000:
                ax.scatter([2] * len(self.fraud_df[feature]), self.fraud_df[feature], 
                          alpha=0.3, color='red', s=10)
            
            ax.set_title(f'{feature}\nCohen\'s d={self.results[feature]["cohens_d"]:.3f} ({self.results[feature]["effect_magnitude"]})')
            ax.set_ylabel('Value')
            ax.grid(True, alpha=0.3, axis='y')
        
        # Hide empty subplots
        for idx in range(len(top_features), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('Feature Comparison: Fraud vs Non-Fraud', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _plot_conditional_distributions(self):
        """
        Plot conditional distributions (feature distributions conditioned on target).
        """
        # Select top features
        sorted_features = sorted(self.results.items(), key=lambda x: x[1]['cohens_d'], reverse=True)
        top_features = [f[0] for f in sorted_features[:9]]
        
        fig, axes = plt.subplots(3, 3, figsize=(18, 15))
        axes = axes.flatten()
        
        for idx, feature in enumerate(top_features):
            ax = axes[idx]
            
            # Plot KDEs
            sns.kdeplot(data=self.non_fraud_df, x=feature, ax=ax, 
                       label='Non-Fraud', color='blue', fill=True, alpha=0.3)
            sns.kdeplot(data=self.fraud_df, x=feature, ax=ax, 
                       label='Fraud', color='red', fill=True, alpha=0.3)
            
            ax.set_title(f'{feature}\nKS Test p-value: {self.results[feature]["ks_p_value"]:.4e}')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.suptitle('Conditional Distributions: Fraud vs Non-Fraud', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _plot_statistical_tests(self):
        """
        Visualize results of statistical tests.
        """
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Effect sizes
        features = list(self.results.keys())
        effect_sizes = [self.results[f]['cohens_d'] for f in features]
        
        effect_df = pd.DataFrame({'Feature': features, 'Cohen\'s d': effect_sizes})
        effect_df = effect_df.sort_values("Cohen's d", ascending=False).head(20)
        
        colors = ['red' if d >= 0.8 else 'orange' if d >= 0.5 else 'green' 
                 for d in effect_df["Cohen's d"]]
        
        axes[0, 0].barh(range(len(effect_df)), effect_df["Cohen's d"], color=colors)
        axes[0, 0].set_yticks(range(len(effect_df)))
        axes[0, 0].set_yticklabels(effect_df['Feature'])
        axes[0, 0].set_xlabel("Cohen's d")
        axes[0, 0].set_title('Top 20 Features by Effect Size', fontweight='bold')
        axes[0, 0].axvline(x=0.2, color='gray', linestyle='--', alpha=0.5)
        axes[0, 0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
        axes[0, 0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.5)
        axes[0, 0].grid(True, alpha=0.3, axis='x')
        
        # 2. P-values (-log10 scale)
        p_values = [-np.log10(self.results[f]['t_p_value'] + 1e-10) for f in features]
        p_df = pd.DataFrame({'Feature': features, '-log10(p-value)': p_values})
        p_df = p_df.sort_values('-log10(p-value)', ascending=False).head(20)
        
        axes[0, 1].barh(range(len(p_df)), p_df['-log10(p-value)'], color='skyblue')
        axes[0, 1].set_yticks(range(len(p_df)))
        axes[0, 1].set_yticklabels(p_df['Feature'])
        axes[0, 1].set_xlabel('-log10(p-value)')
        axes[0, 1].set_title('Top 20 Features by Statistical Significance', fontweight='bold')
        axes[0, 1].axvline(x=-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3, axis='x')
        
        # 3. Information Value
        iv_values = [self.results[f]['information_value'] for f in features]
        iv_df = pd.DataFrame({'Feature': features, 'Information Value': iv_values})
        iv_df = iv_df.sort_values('Information Value', ascending=False).head(20)
        
        colors = ['red' if iv > 0.5 else 'orange' if iv > 0.3 else 'green' 
                 for iv in iv_df['Information Value']]
        
        axes[1, 0].barh(range(len(iv_df)), iv_df['Information Value'], color=colors)
        axes[1, 0].set_yticks(range(len(iv_df)))
        axes[1, 0].set_yticklabels(iv_df['Feature'])
        axes[1, 0].set_xlabel('Information Value')
        axes[1, 0].set_title('Top 20 Features by Information Value', fontweight='bold')
        axes[1, 0].axvline(x=0.3, color='orange', linestyle='--', alpha=0.5)
        axes[1, 0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
        axes[1, 0].grid(True, alpha=0.3, axis='x')
        
        # 4. Summary statistics
        axes[1, 1].axis('off')
        summary_text = f"""
        BIVARIATE ANALYSIS SUMMARY
        
        Total features analyzed: {len(features)}
        
        Features with significant differences (p < 0.05): 
        {sum(1 for f in features if self.results[f]['significant'])}
        
        Effect Size Distribution:
        - Large (d ≥ 0.8): {sum(1 for f in features if self.results[f]['cohens_d'] >= 0.8)}
        - Medium (0.5 ≤ d < 0.8): {sum(1 for f in features if 0.5 <= self.results[f]['cohens_d'] < 0.8)}
        - Small (0.2 ≤ d < 0.5): {sum(1 for f in features if 0.2 <= self.results[f]['cohens_d'] < 0.5)}
        - Negligible (d < 0.2): {sum(1 for f in features if self.results[f]['cohens_d'] < 0.2)}
        
        Information Value Distribution:
        - Strong (IV > 0.5): {sum(1 for f in features if self.results[f]['information_value'] > 0.5)}
        - Medium (0.3 < IV ≤ 0.5): {sum(1 for f in features if 0.3 < self.results[f]['information_value'] <= 0.5)}
        - Weak (IV ≤ 0.3): {sum(1 for f in features if self.results[f]['information_value'] <= 0.3)}
        """
        
        axes[1, 1].text(0.1, 0.9, summary_text, transform=axes[1, 1].transAxes,
                       fontsize=12, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle('Statistical Test Results Summary', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Perform bivariate analysis
bivariate_analyzer = BivariateAnalyzer(df)
bivariate_results = bivariate_analyzer.analyze_all_features()

# %% [markdown]
# ### 5. Correlation Analysis

# %%
class CorrelationAnalyzer:
    """
    Comprehensive correlation analysis.
    """
    
    def __init__(self, df, target_col='Class'):
        self.df = df
        self.target_col = target_col
        self.results = {}
        
    def analyze_correlations(self):
        """
        Perform correlation analysis using multiple methods.
        """
        print("\n" + "="*60)
        print("CORRELATION ANALYSIS")
        print("="*60)
        
        # Get numerical features
        self.numerical_cols = self.df.select_dtypes(include=[np.number]).columns
        self.numerical_cols = [col for col in self.numerical_cols if col != self.target_col]
        
        # 1. Pearson Correlation (linear)
        self._pearson_correlation()
        
        # 2. Spearman Correlation (monotonic)
        self._spearman_correlation()
        
        # 3. Kendall Tau Correlation (ordinal)
        self._kendall_correlation()
        
        # 4. Mutual Information (non-linear)
        self._mutual_information()
        
        # 5. Distance Correlation (non-linear dependence)
        self._distance_correlation()
        
        # Create visualizations
        self._plot_correlation_heatmaps()
        self._plot_correlation_comparison()
        self._plot_scatter_matrix()
        
        return self.results
    
    def _pearson_correlation(self):
        """
        Calculate Pearson correlation coefficients.
        """
        # Calculate correlation matrix
        corr_matrix = self.df[self.numerical_cols + [self.target_col]].corr(method='pearson')
        
        # Extract correlations with target
        target_corr = corr_matrix[self.target_col].drop(self.target_col)
        
        self.results['pearson'] = {
            'matrix': corr_matrix,
            'target_corr': target_corr,
            'top_positive': target_corr.nlargest(10),
            'top_negative': target_corr.nsmallest(10)
        }
        
        print("\n📊 Pearson Correlation (Linear):")
        print(f"  Top 5 positive correlations: {target_corr.nlargest(5).to_dict()}")
        print(f"  Top 5 negative correlations: {target_corr.nsmallest(5).to_dict()}")
    
    def _spearman_correlation(self):
        """
        Calculate Spearman rank correlation coefficients.
        """
        # Calculate correlation matrix
        corr_matrix = self.df[self.numerical_cols + [self.target_col]].corr(method='spearman')
        
        # Extract correlations with target
        target_corr = corr_matrix[self.target_col].drop(self.target_col)
        
        self.results['spearman'] = {
            'matrix': corr_matrix,
            'target_corr': target_corr,
            'top_positive': target_corr.nlargest(10),
            'top_negative': target_corr.nsmallest(10)
        }
        
        print("\n📊 Spearman Correlation (Monotonic):")
        print(f"  Top 5 positive correlations: {target_corr.nlargest(5).to_dict()}")
        print(f"  Top 5 negative correlations: {target_corr.nsmallest(5).to_dict()}")
    
    def _kendall_correlation(self):
        """
        Calculate Kendall Tau correlation coefficients.
        """
        # Calculate correlation matrix (using a sample for efficiency)
        sample_size = min(10000, len(self.df))
        sample_df = self.df[self.numerical_cols + [self.target_col]].sample(sample_size, random_state=42)
        
        corr_matrix = sample_df.corr(method='kendall')
        
        # Extract correlations with target
        target_corr = corr_matrix[self.target_col].drop(self.target_col)
        
        self.results['kendall'] = {
            'matrix': corr_matrix,
            'target_corr': target_corr,
            'top_positive': target_corr.nlargest(10),
            'top_negative': target_corr.nsmallest(10)
        }
        
        print("\n📊 Kendall Tau Correlation (Ordinal):")
        print(f"  Top 5 positive correlations: {target_corr.nlargest(5).to_dict()}")
        print(f"  Top 5 negative correlations: {target_corr.nsmallest(5).to_dict()}")
    
    def _mutual_information(self):
        """
        Calculate mutual information between features and target.
        """
        from sklearn.feature_selection import mutual_info_classif
        
        # Calculate mutual information
        mi_scores = mutual_info_classif(
            self.df[self.numerical_cols], 
            self.df[self.target_col],
            random_state=42,
            n_neighbors=3
        )
        
        mi_series = pd.Series(mi_scores, index=self.numerical_cols)
        mi_series = mi_series.sort_values(ascending=False)
        
        self.results['mutual_info'] = {
            'scores': mi_series,
            'top_features': mi_series.head(10)
        }
        
        print("\n📊 Mutual Information (Non-linear):")
        print(f"  Top 5 features: {mi_series.head(5).to_dict()}")
    
    def _distance_correlation(self):
        """
        Calculate distance correlation (measures both linear and non-linear dependence).
        """
        try:
            import dcor  # distance correlation library
            
            # Calculate for top features (computationally expensive)
            sample_size = min(5000, len(self.df))
            sample_df = self.df[self.numerical_cols[:10] + [self.target_col]].sample(sample_size, random_state=42)
            
            dcor_scores = {}
            for col in self.numerical_cols[:10]:
                dcor_scores[col] = dcor.distance_correlation(
                    sample_df[col].values.reshape(-1, 1),
                    sample_df[self.target_col].values.reshape(-1, 1)
                )
            
            dcor_series = pd.Series(dcor_scores).sort_values(ascending=False)
            
            self.results['distance_corr'] = {
                'scores': dcor_series,
                'top_features': dcor_series.head(5)
            }
            
            print("\n📊 Distance Correlation (General Dependence):")
            print(f"  Top 5 features: {dcor_series.head(5).to_dict()}")
            
        except ImportError:
            print("\n⚠️  Distance correlation not available (install dcor package)")
            self.results['distance_corr'] = None
    
    def _plot_correlation_heatmaps(self):
        """
        Create correlation heatmaps.
        """
        fig, axes = plt.subplots(2, 2, figsize=(18, 16))
        
        # 1. Pearson correlation heatmap
        if 'pearson' in self.results:
            corr_matrix = self.results['pearson']['matrix']
            
            # Select top features for clarity
            top_features = self.results['pearson']['target_corr'].abs().nlargest(15).index.tolist()
            top_features = [f for f in top_features if f in corr_matrix.columns]
            
            sns.heatmap(corr_matrix.loc[top_features + [self.target_col], top_features + [self.target_col]],
                       annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       square=True, linewidths=1, ax=axes[0, 0],
                       cbar_kws={"shrink": 0.8})
            axes[0, 0].set_title('Pearson Correlation (Linear)', fontweight='bold')
        
        # 2. Spearman correlation heatmap
        if 'spearman' in self.results:
            corr_matrix = self.results['spearman']['matrix']
            
            sns.heatmap(corr_matrix.loc[top_features + [self.target_col], top_features + [self.target_col]],
                       annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       square=True, linewidths=1, ax=axes[0, 1],
                       cbar_kws={"shrink": 0.8})
            axes[0, 1].set_title('Spearman Correlation (Monotonic)', fontweight='bold')
        
        # 3. Mutual Information bar plot
        if 'mutual_info' in self.results:
            mi_scores = self.results['mutual_info']['scores'].head(20)
            axes[1, 0].barh(range(len(mi_scores)), mi_scores.values, color='skyblue')
            axes[1, 0].set_yticks(range(len(mi_scores)))
            axes[1, 0].set_yticklabels(mi_scores.index)
            axes[1, 0].set_xlabel('Mutual Information')
            axes[1, 0].set_title('Mutual Information with Target', fontweight='bold')
            axes[1, 0].grid(True, alpha=0.3, axis='x')
        
        # 4. Correlation comparison
        if 'pearson' in self.results and 'spearman' in self.results:
            pearson_corr = self.results['pearson']['target_corr']
            spearman_corr = self.results['spearman']['target_corr']
            
            comparison_df = pd.DataFrame({
                'Pearson': pearson_corr,
                'Spearman': spearman_corr
            }).dropna()
            
            axes[1, 1].scatter(comparison_df['Pearson'], comparison_df['Spearman'], alpha=0.6)
            
            # Add diagonal line
            min_val = min(comparison_df.min().min(), -0.1)
            max_val = max(comparison_df.max().max(), 0.1)
            axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5)
            
            # Annotate outliers (features with large difference between correlations)
            comparison_df['diff'] = abs(comparison_df['Pearson'] - comparison_df['Spearman'])
            outliers = comparison_df.nlargest(5, 'diff')
            
            for idx, row in outliers.iterrows():
                axes[1, 1].annotate(idx, (row['Pearson'], row['Spearman']), 
                                   fontsize=9, ha='center')
            
            axes[1, 1].set_xlabel('Pearson Correlation')
            axes[1, 1].set_ylabel('Spearman Correlation')
            axes[1, 1].set_title('Pearson vs Spearman Correlation', fontweight='bold')
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.suptitle('Correlation Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _plot_correlation_comparison(self):
        """
        Compare different correlation measures.
        """
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Top correlations by method
        if 'pearson' in self.results:
            top_pearson = self.results['pearson']['target_corr'].abs().nlargest(10)
            axes[0, 0].barh(range(len(top_pearson)), top_pearson.values, color='skyblue')
            axes[0, 0].set_yticks(range(len(top_pearson)))
            axes[0, 0].set_yticklabels(top_pearson.index)
            axes[0, 0].set_xlabel('Absolute Correlation')
            axes[0, 0].set_title('Top 10 Features (Pearson)', fontweight='bold')
            axes[0, 0].grid(True, alpha=0.3, axis='x')
        
        # 2. Top correlations by Spearman
        if 'spearman' in self.results:
            top_spearman = self.results['spearman']['target_corr'].abs().nlargest(10)
            axes[0, 1].barh(range(len(top_spearman)), top_spearman.values, color='lightcoral')
            axes[0, 1].set_yticks(range(len(top_spearman)))
            axes[0, 1].set_yticklabels(top_spearman.index)
            axes[0, 1].set_xlabel('Absolute Correlation')
            axes[0, 1].set_title('Top 10 Features (Spearman)', fontweight='bold')
            axes[0, 1].grid(True, alpha=0.3, axis='x')
        
        # 3. Top features by mutual information
        if 'mutual_info' in self.results:
            top_mi = self.results['mutual_info']['scores'].head(10)
            axes[1, 0].barh(range(len(top_mi)), top_mi.values, color='lightgreen')
            axes[1, 0].set_yticks(range(len(top_mi)))
            axes[1, 0].set_yticklabels(top_mi.index)
            axes[1, 0].set_xlabel('Mutual Information')
            axes[1, 0].set_title('Top 10 Features (Mutual Information)', fontweight='bold')
            axes[1, 0].grid(True, alpha=0.3, axis='x')
        
        # 4. Consistency across methods
        axes[1, 1].axis('off')
        
        # Create a table showing consistency
        if all(k in self.results for k in ['pearson', 'spearman', 'mutual_info']):
            top_features_pearson = set(self.results['pearson']['target_corr'].abs().nlargest(10).index)
            top_features_spearman = set(self.results['spearman']['target_corr'].abs().nlargest(10).index)
            top_features_mi = set(self.results['mutual_info']['scores'].head(10).index)
            
            intersection_all = top_features_pearson & top_features_spearman & top_features_mi
            intersection_ps = top_features_pearson & top_features_spearman
            intersection_pm = top_features_pearson & top_features_mi
            intersection_sm = top_features_spearman & top_features_mi
            
            table_data = [
                ['Metric', 'Count', 'Features'],
                ['All 3 methods', len(intersection_all), ', '.join(list(intersection_all)[:5]) + ('...' if len(intersection_all) > 5 else '')],
                ['Pearson & Spearman', len(intersection_ps), ', '.join(list(intersection_ps)[:5]) + ('...' if len(intersection_ps) > 5 else '')],
                ['Pearson & MI', len(intersection_pm), ', '.join(list(intersection_pm)[:5]) + ('...' if len(intersection_pm) > 5 else '')],
                ['Spearman & MI', len(intersection_sm), ', '.join(list(intersection_sm)[:5]) + ('...' if len(intersection_sm) > 5 else '')]
            ]
            
            table = axes[1, 1].table(cellText=table_data, loc='center', cellLoc='left')
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1, 1.5)
            
            # Style the header
            for (i, j), cell in table.get_celld().items():
                if i == 0:
                    cell.set_facecolor('#40466e')
                    cell.set_text_props(color='white', fontweight='bold')
            
            axes[1, 1].set_title('Feature Selection Consistency', fontweight='bold', pad=20)
        
        plt.suptitle('Correlation Measures Comparison', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _plot_scatter_matrix(self):
        """
        Create scatter matrix for top features.
        """
        # Select top features for scatter matrix
        if 'pearson' in self.results:
            top_features = self.results['pearson']['target_corr'].abs().nlargest(5).index.tolist()
            
            # Add target
            plot_features = top_features + [self.target_col]
            
            # Create scatter matrix
            fig = plt.figure(figsize=(15, 15))
            
            # Use different colors for fraud/non-fraud
            colors = ['blue' if c == 0 else 'red' for c in self.df[self.target_col]]
            
            # Create pairplot
            pd.plotting.scatter_matrix(
                self.df[plot_features].sample(min(5000, len(self.df))),
                figsize=(15, 15),
                diagonal='hist',
                color=colors,
                alpha=0.3
            )
            
            plt.suptitle('Scatter Matrix - Top 5 Features', fontsize=16, fontweight='bold', y=1.02)
            plt.tight_layout()
            plt.show()

# Perform correlation analysis
correlation_analyzer = CorrelationAnalyzer(df)
correlation_results = correlation_analyzer.analyze_correlations()

# %% [markdown]
# ### 6. Multivariate Analysis

# %%
class MultivariateAnalyzer:
    """
    Multivariate analysis using dimensionality reduction techniques.
    """
    
    def __init__(self, df, target_col='Class'):
        self.df = df
        self.target_col = target_col
        self.results = {}
        
    def analyze_multivariate(self):
        """
        Perform multivariate analysis using PCA and t-SNE.
        """
        print("\n" + "="*60)
        print("MULTIVARIATE ANALYSIS")
        print("="*60)
        
        # Prepare data
        self._prepare_data()
        
        # PCA Analysis
        self._pca_analysis()
        
        # t-SNE Analysis
        self._tsne_analysis()
        
        # Feature importance using Random Forest
        self._feature_importance_rf()
        
        return self.results
    
    def _prepare_data(self):
        """
        Prepare data for multivariate analysis.
        """
        # Get numerical features
        self.numerical_cols = self.df.select_dtypes(include=[np.number]).columns
        self.numerical_cols = [col for col in self.numerical_cols if col != self.target_col]
        
        # Standardize data
        from sklearn.preprocessing import StandardScaler
        
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.df[self.numerical_cols])
        self.y = self.df[self.target_col].values
        
        print(f"\n✅ Data prepared: {self.X_scaled.shape[0]} samples, {self.X_scaled.shape[1]} features")
    
    def _pca_analysis(self):
        """
        Perform PCA analysis.
        """
        from sklearn.decomposition import PCA
        
        # Fit PCA
        self.pca = PCA()
        X_pca = self.pca.fit_transform(self.X_scaled)
        
        # Store results
        self.results['pca'] = {
            'explained_variance_ratio': self.pca.explained_variance_ratio_,
            'cumulative_variance': np.cumsum(self.pca.explained_variance_ratio_),
            'components': self.pca.components_,
            'transformed_data': X_pca[:, :2]  # First 2 components for visualization
        }
        
        print("\n📊 PCA Analysis:")
        print(f"  Variance explained by PC1: {self.pca.explained_variance_ratio_[0]:.4f}")
        print(f"  Variance explained by PC2: {self.pca.explained_variance_ratio_[1]:.4f}")
        print(f"  Total variance explained (first 2 components): {np.sum(self.pca.explained_variance_ratio_[:2]):.4f}")
        
        # Plot PCA results
        self._plot_pca_results()
    
    def _plot_pca_results(self):
        """
        Plot PCA results.
        """
        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        
        # 1. Scree plot (explained variance)
        n_components = 20
        axes[0, 0].bar(range(1, n_components+1), 
                      self.results['pca']['explained_variance_ratio'][:n_components],
                      alpha=0.7, label='Individual')
        axes[0, 0].plot(range(1, n_components+1), 
                       self.results['pca']['cumulative_variance'][:n_components],
                       'ro-', label='Cumulative')
        axes[0, 0].set_xlabel('Principal Component')
        axes[0, 0].set_ylabel('Explained Variance Ratio')
        axes[0, 0].set_title('PCA Scree Plot', fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Cumulative variance
        axes[0, 1].plot(range(1, len(self.results['pca']['cumulative_variance'])+1),
                       self.results['pca']['cumulative_variance'] * 100,
                       'b-', linewidth=2)
        axes[0, 1].axhline(y=95, color='r', linestyle='--', label='95% threshold')
        axes[0, 1].set_xlabel('Number of Components')
        axes[0, 1].set_ylabel('Cumulative Variance Explained (%)')
        axes[0, 1].set_title('Cumulative Explained Variance', fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Find number of components for 95% variance
        n_95 = np.argmax(self.results['pca']['cumulative_variance'] >= 0.95) + 1
        axes[0, 1].axvline(x=n_95, color='g', linestyle='--', 
                          label=f'{n_95} components for 95%')
        axes[0, 1].legend()
        
        # 3. 2D PCA projection
        X_pca_2d = self.results['pca']['transformed_data']
        
        # Separate by class
        fraud_mask = self.y == 1
        non_fraud_mask = self.y == 0
        
        # Plot non-fraud (light blue)
        axes[1, 0].scatter(X_pca_2d[non_fraud_mask, 0], X_pca_2d[non_fraud_mask, 1],
                          c='blue', alpha=0.1, s=5, label='Non-Fraud')
        
        # Plot fraud (red)
        axes[1, 0].scatter(X_pca_2d[fraud_mask, 0], X_pca_2d[fraud_mask, 1],
                          c='red', alpha=0.5, s=20, label='Fraud')
        
        axes[1, 0].set_xlabel(f'PC1 ({self.results["pca"]["explained_variance_ratio"][0]:.2%})')
        axes[1, 0].set_ylabel(f'PC2 ({self.results["pca"]["explained_variance_ratio"][1]:.2%})')
        axes[1, 0].set_title('PCA Projection (2D)', fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Feature contributions to PC1
        pc1_loadings = pd.Series(self.results['pca']['components'][0], 
                                 index=self.numerical_cols)
        top_pc1 = pc1_loadings.abs().nlargest(10)
        
        colors = ['red' if val < 0 else 'green' for val in pc1_loadings[top_pc1.index]]
        
        axes[1, 1].barh(range(len(top_pc1)), top_pc1.values, color=colors)
        axes[1, 1].set_yticks(range(len(top_pc1)))
        axes[1, 1].set_yticklabels(top_pc1.index)
        axes[1, 1].set_xlabel('Loading')
        axes[1, 1].set_title('Top 10 Feature Contributions to PC1', fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3, axis='x')
        
        plt.suptitle('Principal Component Analysis (PCA) Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _tsne_analysis(self):
        """
        Perform t-SNE analysis.
        """
        from sklearn.manifold import TSNE
        
        # Use a sample for t-SNE (computationally expensive)
        sample_size = min(5000, len(self.df))
        sample_indices = np.random.choice(len(self.df), sample_size, replace=False)
        
        X_sample = self.X_scaled[sample_indices]
        y_sample = self.y[sample_indices]
        
        # Fit t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
        X_tsne = tsne.fit_transform(X_sample)
        
        self.results['tsne'] = {
            'transformed_data': X_tsne,
            'labels': y_sample
        }
        
        print("\n📊 t-SNE Analysis:")
        print(f"  Sample size: {sample_size}")
        print(f"  Perplexity: 30")
        print(f"  Final KL divergence: {tsne.kl_divergence_:.4f}")
        
        # Plot t-SNE results
        self._plot_tsne_results()
    
    def _plot_tsne_results(self):
        """
        Plot t-SNE results.
        """
        if 'tsne' not in self.results:
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        X_tsne = self.results['tsne']['transformed_data']
        y_sample = self.results['tsne']['labels']
        
        # Separate by class
        fraud_mask = y_sample == 1
        non_fraud_mask = y_sample == 0
        
        # 1. t-SNE projection with class colors
        axes[0].scatter(X_tsne[non_fraud_mask, 0], X_tsne[non_fraud_mask, 1],
                       c='blue', alpha=0.3, s=10, label='Non-Fraud')
        axes[0].scatter(X_tsne[fraud_mask, 0], X_tsne[fraud_mask, 1],
                       c='red', alpha=0.7, s=30, label='Fraud', edgecolors='black', linewidth=0.5)
        
        axes[0].set_xlabel('t-SNE Component 1')
        axes[0].set_ylabel('t-SNE Component 2')
        axes[0].set_title('t-SNE Projection (Colored by Class)', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. Density contours
        from scipy.stats import gaussian_kde
        
        # Calculate point density for non-fraud
        if non_fraud_mask.sum() > 100:
            xy_nonfraud = np.vstack([X_tsne[non_fraud_mask, 0], X_tsne[non_fraud_mask, 1]])
            z_nonfraud = gaussian_kde(xy_nonfraud)(xy_nonfraud)
            
            # Sort points by density for better visualization
            idx_nonfraud = z_nonfraud.argsort()
            x_nonfraud_sorted = X_tsne[non_fraud_mask, 0][idx_nonfraud]
            y_nonfraud_sorted = X_tsne[non_fraud_mask, 1][idx_nonfraud]
            z_nonfraud_sorted = z_nonfraud[idx_nonfraud]
            
            scatter2 = axes[1].scatter(x_nonfraud_sorted, y_nonfraud_sorted,
                                      c=z_nonfraud_sorted, cmap='Blues',
                                      s=10, alpha=0.5, label='Non-Fraud')
        
        # Calculate point density for fraud
        if fraud_mask.sum() > 10:
            xy_fraud = np.vstack([X_tsne[fraud_mask, 0], X_tsne[fraud_mask, 1]])
            z_fraud = gaussian_kde(xy_fraud)(xy_fraud)
            
            # Sort points by density
            idx_fraud = z_fraud.argsort()
            x_fraud_sorted = X_tsne[fraud_mask, 0][idx_fraud]
            y_fraud_sorted = X_tsne[fraud_mask, 1][idx_fraud]
            z_fraud_sorted = z_fraud[idx_fraud]
            
            scatter3 = axes[1].scatter(x_fraud_sorted, y_fraud_sorted,
                                      c=z_fraud_sorted, cmap='Reds',
                                      s=30, alpha=0.7, label='Fraud',
                                      edgecolors='black', linewidth=0.5)
        
        axes[1].set_xlabel('t-SNE Component 1')
        axes[1].set_ylabel('t-SNE Component 2')
        axes[1].set_title('t-SNE Projection (Density-based Coloring)', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.suptitle('t-SNE Visualization', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _feature_importance_rf(self):
        """
        Calculate feature importance using Random Forest.
        """
        from sklearn.ensemble import RandomForestClassifier
        
        # Use a sample for efficiency
        sample_size = min(50000, len(self.df))
        sample_indices = np.random.choice(len(self.df), sample_size, replace=False)
        
        X_sample = self.X_scaled[sample_indices]
        y_sample = self.y[sample_indices]
        
        # Train Random Forest
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        
        rf.fit(X_sample, y_sample)
        
        # Get feature importance
        importance = rf.feature_importances_
        importance_series = pd.Series(importance, index=self.numerical_cols)
        importance_series = importance_series.sort_values(ascending=False)
        
        self.results['rf_importance'] = {
            'importance': importance_series,
            'model': rf,
            'top_features': importance_series.head(20)
        }
        
        print("\n📊 Random Forest Feature Importance:")
        print(f"  Top 5 features: {importance_series.head(5).to_dict()}")
        
        # Plot feature importance
        self._plot_feature_importance()
    
    def _plot_feature_importance(self):
        """
        Plot feature importance from Random Forest.
        """
        if 'rf_importance' not in self.results:
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        
        importance_series = self.results['rf_importance']['importance']
        top_20 = importance_series.head(20)
        
        # 1. Bar plot of top 20 features
        axes[0].barh(range(len(top_20)), top_20.values, color='skyblue')
        axes[0].set_yticks(range(len(top_20)))
        axes[0].set_yticklabels(top_20.index)
        axes[0].set_xlabel('Importance')
        axes[0].set_title('Top 20 Features - Random Forest Importance', fontweight='bold')
        axes[0].invert_yaxis()  # Highest at top
        axes[0].grid(True, alpha=0.3, axis='x')
        
        # 2. Cumulative importance
        sorted_importance = importance_series.values
        cumulative_importance = np.cumsum(sorted_importance)
        
        axes[1].plot(range(1, len(cumulative_importance)+1), 
                    cumulative_importance * 100, 'b-', linewidth=2)
        axes[1].axhline(y=95, color='r', linestyle='--', label='95% threshold')
        axes[1].set_xlabel('Number of Features')
        axes[1].set_ylabel('Cumulative Importance (%)')
        axes[1].set_title('Cumulative Feature Importance', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Find number of features for 95% importance
        n_95 = np.argmax(cumulative_importance >= 0.95) + 1
        axes[1].axvline(x=n_95, color='g', linestyle='--', 
                       label=f'{n_95} features for 95%')
        axes[1].legend()
        
        plt.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Perform multivariate analysis
multivariate_analyzer = MultivariateAnalyzer(df)
multivariate_results = multivariate_analyzer.analyze_multivariate()

# %% [markdown]
# ### 7. Temporal Pattern Analysis

# %%
class TemporalAnalyzer:
    """
    Analyze temporal patterns in fraud data.
    """
    
    def __init__(self, df):
        self.df = df
        self.results = {}
        
    def analyze_temporal_patterns(self):
        """
        Analyze temporal patterns in transactions.
        """
        print("\n" + "="*60)
        print("TEMPORAL PATTERN ANALYSIS")
        print("="*60)
        
        # Create time-based features
        self._create_time_features()
        
        # Analyze hourly patterns
        self._analyze_hourly_patterns()
        
        # Analyze daily patterns
        self._analyze_daily_patterns()
        
        # Analyze transaction velocity
        self._analyze_transaction_velocity()
        
        # Analyze time gaps
        self._analyze_time_gaps()
        
        return self.results
    
    def _create_time_features(self):
        """
        Create time-based features from the Time column.
        """
        # Convert seconds to hours
        self.df['hour'] = (self.df['Time'] / 3600) % 24
        
        # Create cyclical features for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24)
        
        # Day of week (assuming Time starts from day 0)
        self.df['day'] = (self.df['Time'] / (3600 * 24)) % 7
        self.df['is_weekend'] = (self.df['day'] >= 5).astype(int)
        
        print("\n✅ Created temporal features: hour, hour_sin, hour_cos, day, is_weekend")
    
    def _analyze_hourly_patterns(self):
        """
        Analyze fraud patterns by hour of day.
        """
        # Calculate fraud rate by hour
        hourly_stats = self.df.groupby('hour').agg({
            'Class': ['count', 'sum', 'mean']
        }).round(4)
        
        hourly_stats.columns = ['total_transactions', 'fraud_count', 'fraud_rate']
        hourly_stats = hourly_stats.reset_index()
        
        # Calculate additional metrics
        hourly_stats['fraud_rate_pct'] = hourly_stats['fraud_rate'] * 100
        
        self.results['hourly'] = hourly_stats.to_dict('records')
        
        print("\n📊 Hourly Pattern Analysis:")
        print(f"  Peak fraud hour: {hourly_stats.loc[hourly_stats['fraud_count'].idxmax(), 'hour']:.0f}")
        print(f"  Peak fraud rate: {hourly_stats['fraud_rate'].max()*100:.4f}%")
        print(f"  Lowest fraud rate: {hourly_stats['fraud_rate'].min()*100:.4f}%")
        
        # Visualize hourly patterns
        self._plot_hourly_patterns(hourly_stats)
    
    def _plot_hourly_patterns(self, hourly_stats):
        """
        Plot hourly patterns.
        """
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Transaction volume by hour
        axes[0, 0].bar(hourly_stats['hour'], hourly_stats['total_transactions'], 
                       color='skyblue', edgecolor='black', alpha=0.7)
        axes[0, 0].set_xlabel('Hour of Day')
        axes[0, 0].set_ylabel('Number of Transactions')
        axes[0, 0].set_title('Transaction Volume by Hour', fontweight='bold')
        axes[0, 0].grid(True, alpha=0.3, axis='y')
        
        # 2. Fraud count by hour
        axes[0, 1].bar(hourly_stats['hour'], hourly_stats['fraud_count'], 
                       color='red', edgecolor='black', alpha=0.7)
        axes[0, 1].set_xlabel('Hour of Day')
        axes[0, 1].set_ylabel('Fraud Count')
        axes[0, 1].set_title('Fraud Count by Hour', fontweight='bold')
        axes[0, 1].grid(True, alpha=0.3, axis='y')
        
        # 3. Fraud rate by hour
        axes[1, 0].plot(hourly_stats['hour'], hourly_stats['fraud_rate_pct'], 
                        'ro-', linewidth=2, markersize=8)
        axes[1, 0].fill_between(hourly_stats['hour'], hourly_stats['fraud_rate_pct'], 
                                 alpha=0.3, color='red')
        axes[1, 0].axhline(y=hourly_stats['fraud_rate_pct'].mean(), 
                            color='blue', linestyle='--', 
                            label=f"Mean: {hourly_stats['fraud_rate_pct'].mean():.4f}%")
        axes[1, 0].set_xlabel('Hour of Day')
        axes[1, 0].set_ylabel('Fraud Rate (%)')
        axes[1, 0].set_title('Fraud Rate by Hour', fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Radar chart for hourly fraud rate
        from math import pi
        
        # Prepare data for radar chart
        categories = [f"{int(h):02d}:00" for h in range(0, 24, 3)]
        values = [hourly_stats[hourly_stats['hour'] == h]['fraud_rate_pct'].values[0] 
                 for h in range(0, 24, 3)]
        
        # Repeat first value to close the circle
        values += values[:1]
        angles = [n / len(categories) * 2 * pi for n in range(len(categories))]
        angles += angles[:1]
        
        ax = plt.subplot(2, 2, 4, projection='polar')
        ax.plot(angles, values, 'o-', linewidth=2, color='red')
        ax.fill(angles, values, alpha=0.25, color='red')
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories)
        ax.set_title('Fraud Rate by Hour (Radar)', fontweight='bold', pad=20)
        
        plt.suptitle('Hourly Fraud Pattern Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _analyze_daily_patterns(self):
        """
        Analyze fraud patterns by day of week.
        """
        day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 
                    'Friday', 'Saturday', 'Sunday']
        
        daily_stats = self.df.groupby('day').agg({
            'Class': ['count', 'sum', 'mean']
        }).round(4)
        
        daily_stats.columns = ['total_transactions', 'fraud_count', 'fraud_rate']
        daily_stats = daily_stats.reset_index()
        daily_stats['day_name'] = daily_stats['day'].map(lambda x: day_names[int(x)])
        daily_stats['fraud_rate_pct'] = daily_stats['fraud_rate'] * 100
        
        self.results['daily'] = daily_stats.to_dict('records')
        
        print("\n📊 Daily Pattern Analysis:")
        print(f"  Peak fraud day: {daily_stats.loc[daily_stats['fraud_count'].idxmax(), 'day_name']}")
        print(f"  Weekend fraud rate: {daily_stats[daily_stats['day'] >= 5]['fraud_rate'].mean()*100:.4f}%")
        print(f"  Weekday fraud rate: {daily_stats[daily_stats['day'] < 5]['fraud_rate'].mean()*100:.4f}%")
        
        # Visualize daily patterns
        self._plot_daily_patterns(daily_stats)
    
    def _plot_daily_patterns(self, daily_stats):
        """
        Plot daily patterns.
        """
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # 1. Fraud count by day
        colors = ['lightcoral' if d >= 5 else 'skyblue' for d in daily_stats['day']]
        
        axes[0].bar(daily_stats['day_name'], daily_stats['fraud_count'], 
                    color=colors, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Day of Week')
        axes[0].set_ylabel('Fraud Count')
        axes[0].set_title('Fraud Count by Day of Week', fontweight='bold')
        axes[0].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(daily_stats['fraud_count']):
            axes[0].text(i, v + 0.5, str(int(v)), ha='center', va='bottom', fontweight='bold')
        
        # 2. Fraud rate by day
        axes[1].bar(daily_stats['day_name'], daily_stats['fraud_rate_pct'], 
                    color=colors, edgecolor='black', alpha=0.7)
        axes[1].axhline(y=daily_stats['fraud_rate_pct'].mean(), 
                         color='red', linestyle='--', 
                         label=f"Week Avg: {daily_stats['fraud_rate_pct'].mean():.4f}%")
        axes[1].set_xlabel('Day of Week')
        axes[1].set_ylabel('Fraud Rate (%)')
        axes[1].set_title('Fraud Rate by Day of Week', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(daily_stats['fraud_rate_pct']):
            axes[1].text(i, v + 0.005, f'{v:.4f}%', ha='center', va='bottom', fontsize=9)
        
        plt.suptitle('Daily Fraud Pattern Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def _analyze_transaction_velocity(self):
        """
        Analyze transaction velocity patterns.
        """
        # Sort by time to calculate velocity
        df_sorted = self.df.sort_values('Time')
        
        # Calculate transactions per minute for each customer
        # Note: In real data, we'd need customer_id. Here we'll use overall velocity
        
        # Create time windows
        time_windows = [5, 15, 30, 60, 120, 360, 720]  # minutes
        
        velocity_stats = []
        for window in time_windows:
            window_seconds = window * 60
            
            # Count transactions in rolling window
            # This is approximate without customer segmentation
            df_sorted[f'tx_{window}min'] = df_sorted.rolling(
                f'{window_seconds}S', on='Time'
            )['Class'].count()
            
            # Analyze fraud rate by velocity
            velocity_analysis = df_sorted.groupby(pd.cut(df_sorted[f'tx_{window}min'], 
                                                         bins=10)).agg({
                'Class': ['count', 'mean']
            })
            
            velocity_stats.append({
                'window_minutes': window,
                'correlation': df_sorted[f'tx_{window}min'].corr(df_sorted['Class']),
                'max_velocity': df_sorted[f'tx_{window}min'].max()
            })
        
        self.results['velocity'] = velocity_stats
        
        print("\n📊 Transaction Velocity Analysis:")
        for stat in velocity_stats:
            print(f"  {stat['window_minutes']}min window correlation: {stat['correlation']:.4f}")
    
    def _analyze_time_gaps(self):
        """
        Analyze time gaps between transactions.
        """
        # Sort by time
        df_sorted = self.df.sort_values('Time')
        
        # Calculate time gaps
        df_sorted['time_gap'] = df_sorted['Time'].diff()
        
        # Analyze gaps by class
        fraud_gaps = df_sorted[df_sorted['Class'] == 1]['time_gap'].dropna()
        non_fraud_gaps = df_sorted[df_sorted['Class'] == 0]['time_gap'].dropna()
        
        # Statistical comparison
        from scipy import stats
        
        if len(fraud_gaps) > 0 and len(non_fraud_gaps) > 0:
            t_stat, p_value = stats.ttest_ind(fraud_gaps, non_fraud_gaps)
            
            self.results['time_gaps'] = {
                'fraud_mean_gap': fraud_gaps.mean(),
                'fraud_median_gap': fraud_gaps.median(),
                'non_fraud_mean_gap': non_fraud_gaps.mean(),
                'non_fraud_median_gap': non_fraud_gaps.median(),
                't_statistic': t_stat,
                'p_value': p_value,
                'significant': p_value < 0.05
            }
            
            print("\n📊 Time Gap Analysis:")
            print(f"  Fraud mean gap: {fraud_gaps.mean():.2f} seconds")
            print(f"  Non-fraud mean gap: {non_fraud_gaps.mean():.2f} seconds")
            print(f"  T-test p-value: {p_value:.4e}")
            print(f"  Significant difference: {'Yes' if p_value < 0.05 else 'No'}")
            
            # Plot time gaps
            self._plot_time_gaps(fraud_gaps, non_fraud_gaps)
    
    def _plot_time_gaps(self, fraud_gaps, non_fraud_gaps):
        """
        Plot time gap distributions.
        """
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # 1. Histogram of time gaps
        axes[0].hist(non_fraud_gaps / 60, bins=50, alpha=0.7, 
                     label='Non-Fraud', color='blue', density=True)
        axes[0].hist(fraud_gaps / 60, bins=50, alpha=0.7, 
                     label='Fraud', color='red', density=True)
        axes[0].set_xlabel('Time Gap (minutes)')
        axes[0].set_ylabel('Density')
        axes[0].set_title('Time Gap Distribution by Class', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. Boxplot of time gaps
        box_data = [non_fraud_gaps / 60, fraud_gaps / 60]
        bp = axes[1].boxplot(box_data, labels=['Non-Fraud', 'Fraud'], patch_artist=True)
        bp['boxes'][0].set_facecolor('blue')
        bp['boxes'][1].set_facecolor('red')
        axes[1].set_ylabel('Time Gap (minutes)')
        axes[1].set_title('Time Gap Boxplot by Class', fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='y')
        
        # Set y-scale to log for better visualization
        axes[1].set_yscale('log')
        
        plt.suptitle('Time Gap Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Perform temporal analysis
temporal_analyzer = TemporalAnalyzer(df)
temporal_results = temporal_analyzer.analyze_temporal_patterns()

# %% [markdown]
# ### 8. Summary and Feature Selection

# %%
class FeatureSelector:
    """
    Select most important features based on multiple criteria.
    """
    
    def __init__(self, df, target_col='Class'):
        self.df = df
        self.target_col = target_col
        self.results = {}
        
    def select_features(self):
        """
        Select features based on multiple importance measures.
        """
        print("\n" + "="*60)
        print("FEATURE SELECTION")
        print("="*60)
        
        # Get numerical features
        self.numerical_cols = self.df.select_dtypes(include=[np.number]).columns
        self.numerical_cols = [col for col in self.numerical_cols if col != self.target_col]
        
        # Collect importance from various sources
        self._collect_importance_scores()
        
        # Create ensemble importance
        self._create_ensemble_importance()
        
        # Select top features
        self._select_top_features()
        
        # Visualize results
        self._plot_feature_selection()
        
        return self.results
    
    def _collect_importance_scores(self):
        """
        Collect importance scores from different methods.
        """
        self.importance_df = pd.DataFrame(index=self.numerical_cols)
        
        # 1. Effect size from bivariate analysis
        if 'bivariate_results' in globals():
            effect_sizes = {col: bivariate_results[col]['cohens_d'] 
                          for col in self.numerical_cols if col in bivariate_results}
            self.importance_df['effect_size'] = pd.Series(effect_sizes)
        
        # 2. Information value
        if 'bivariate_results' in globals():
            iv_values = {col: bivariate_results[col]['information_value'] 
                        for col in self.numerical_cols if col in bivariate_results}
            self.importance_df['information_value'] = pd.Series(iv_values)
        
        # 3. Pearson correlation
        if 'correlation_results' in globals() and 'pearson' in correlation_results:
            pearson_corr = correlation_results['pearson']['target_corr']
            self.importance_df['pearson_corr'] = pearson_corr.abs()
        
        # 4. Spearman correlation
        if 'correlation_results' in globals() and 'spearman' in correlation_results:
            spearman_corr = correlation_results['spearman']['target_corr']
            self.importance_df['spearman_corr'] = spearman_corr.abs()
        
        # 5. Mutual information
        if 'correlation_results' in globals() and 'mutual_info' in correlation_results:
            mi_scores = correlation_results['mutual_info']['scores']
            self.importance_df['mutual_info'] = mi_scores
        
        # 6. Random Forest importance
        if 'multivariate_results' in globals() and 'rf_importance' in multivariate_results:
            rf_importance = multivariate_results['rf_importance']['importance']
            self.importance_df['rf_importance'] = rf_importance
        
        # Normalize each score to 0-1 range
        for col in self.importance_df.columns:
            min_val = self.importance_df[col].min()
            max_val = self.importance_df[col].max()
            if max_val > min_val:
                self.importance_df[f'{col}_norm'] = (self.importance_df[col] - min_val) / (max_val - min_val)
            else:
                self.importance_df[f'{col}_norm'] = 0
        
        print("\n✅ Collected importance scores from multiple methods")
    
    def _create_ensemble_importance(self):
        """
        Create ensemble importance score.
        """
        # Get normalized columns
        norm_cols = [col for col in self.importance_df.columns if col.endswith('_norm')]
        
        if norm_cols:
            # Simple average of normalized scores
            self.importance_df['ensemble_score'] = self.importance_df[norm_cols].mean(axis=1)
            
            # Weighted average (give more weight to certain methods)
            weights = {
                'effect_size_norm': 0.2,
                'information_value_norm': 0.2,
                'mutual_info_norm': 0.2,
                'rf_importance_norm': 0.2,
                'pearson_corr_norm': 0.1,
                'spearman_corr_norm': 0.1
            }
            
            # Use available columns
            weighted_sum = 0
            total_weight = 0
            for col, weight in weights.items():
                if col in self.importance_df.columns:
                    weighted_sum += self.importance_df[col].fillna(0) * weight
                    total_weight += weight
            
            if total_weight > 0:
                self.importance_df['ensemble_weighted'] = weighted_sum / total_weight
            else:
                self.importance_df['ensemble_weighted'] = self.importance_df['ensemble_score']
            
            # Rank features
            self.importance_df['rank'] = self.importance_df['ensemble_weighted'].rank(ascending=False)
            
            print("\n📊 Top 10 features by ensemble score:")
            top_10 = self.importance_df.nsmallest(10, 'rank')
            for idx, row in top_10.iterrows():
                print(f"  {idx}: score={row['ensemble_weighted']:.4f}")
    
    def _select_top_features(self, n_features=30):
        """
        Select top N features.
        """
        if 'ensemble_weighted' in self.importance_df.columns:
            # Sort by ensemble score
            sorted_features = self.importance_df.sort_values('ensemble_weighted', ascending=False)
            
            # Select top N
            self.top_features = sorted_features.head(n_features).index.tolist()
            
            # Calculate cumulative importance
            sorted_features['cumulative_importance'] = sorted_features['ensemble_weighted'].cumsum() / \
                                                       sorted_features['ensemble_weighted'].sum()
            
            self.results = {
                'all_features': sorted_features.to_dict(),
                'top_features': self.top_features,
                'n_features': n_features,
                'cumulative_importance': sorted_features['cumulative_importance'].to_dict()
            }
            
            print(f"\n✅ Selected top {n_features} features")
            print(f"   These features explain {sorted_features.loc[self.top_features[-1], 'cumulative_importance']*100:.2f}% of total importance")
    
    def _plot_feature_selection(self):
        """
        Visualize feature selection results.
        """
        if 'ensemble_weighted' not in self.importance_df.columns:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(18, 14))
        
        sorted_features = self.importance_df.sort_values('ensemble_weighted', ascending=False)
        
        # 1. Top 20 features bar plot
        top_20 = sorted_features.head(20)
        
        axes[0, 0].barh(range(len(top_20)), top_20['ensemble_weighted'].values, color='skyblue')
        axes[0, 0].set_yticks(range(len(top_20)))
        axes[0, 0].set_yticklabels(top_20.index)
        axes[0, 0].set_xlabel('Ensemble Importance Score')
        axes[0, 0].set_title('Top 20 Features by Ensemble Importance', fontweight='bold')
        axes[0, 0].invert_yaxis()
        axes[0, 0].grid(True, alpha=0.3, axis='x')
        
        # 2. Cumulative importance plot
        axes[0, 1].plot(range(1, len(sorted_features)+1), 
                        sorted_features['cumulative_importance'].values * 100,
                        'b-', linewidth=2)
        axes[0, 1].axhline(y=95, color='r', linestyle='--', label='95% threshold')
        axes[0, 1].axvline(x=len(self.top_features), color='g', linestyle='--',
                           label=f'Selected: {len(self.top_features)} features')
        axes[0, 1].set_xlabel('Number of Features')
        axes[0, 1].set_ylabel('Cumulative Importance (%)')
        axes[0, 1].set_title('Cumulative Importance Curve', fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Feature importance heatmap (top features only)
        if len(self.top_features) > 0:
            # Get correlation matrix of top features
            top_corr = self.df[self.top_features[:15]].corr()
            
            sns.heatmap(top_corr, annot=True, fmt='.2f', cmap='coolwarm',
                       center=0, square=True, linewidths=1, ax=axes[1, 0],
                       cbar_kws={"shrink": 0.8})
            axes[1, 0].set_title('Correlation Matrix of Top 15 Features', fontweight='bold')
        
        # 4. Method agreement radar chart
        if len(self.top_features) > 0:
            # Get scores for top features across methods
            top_features_list = self.top_features[:10]
            methods = ['effect_size_norm', 'information_value_norm', 'mutual_info_norm', 
                      'rf_importance_norm', 'pearson_corr_norm']
            
            # Prepare data for radar chart
            categories = top_features_list
            N = len(categories)
            
            angles = [n / float(N) * 2 * 3.14159 for n in range(N)]
            angles += angles[:1]
            
            ax = plt.subplot(2, 2, 3, projection='polar')
            
            for method in methods:
                if method in self.importance_df.columns:
                    values = self.importance_df.loc[top_features_list, method].values.tolist()
                    values += values[:1]
                    ax.plot(angles, values, 'o-', linewidth=2, label=method.replace('_norm', ''))
                    ax.fill(angles, values, alpha=0.1)
            
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(categories, size=8)
            ax.set_title('Feature Importance by Method (Radar)', fontweight='bold', pad=20)
            ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
        
        # Hide empty subplot
        axes[1, 1].axis('off')
        
        # Add summary text
        summary_text = f"""
        FEATURE SELECTION SUMMARY
        
        Total features analyzed: {len(self.importance_df)}
        Features selected: {len(self.top_features)}
        
        Selection criteria:
        - Effect size (Cohen's d)
        - Information Value
        - Mutual Information
        - Random Forest importance
        - Correlation strength
        
        Next steps:
        - Use selected features for modeling
        - Consider domain expertise for final selection
        - Validate with cross-validation
        """
        
        axes[1, 1].text(0.1, 0.5, summary_text, transform=axes[1, 1].transAxes,
                       fontsize=12, verticalalignment='center',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle('Feature Selection Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Perform feature selection
feature_selector = FeatureSelector(df)
feature_selection_results = feature_selector.select_features()

# %% [markdown]
# ### 9. Save Analysis Results

# %%
def save_analysis_results():
    """
    Save all analysis results to files.
    """
    print("\n" + "="*60)
    print("SAVING ANALYSIS RESULTS")
    print("="*60)
    
    # Create directory
    os.makedirs('../artifacts/reports', exist_ok=True)
    
    # Combine all results
    all_results = {
        'univariate': univariate_results if 'univariate_results' in dir() else {},
        'bivariate': bivariate_results if 'bivariate_results' in dir() else {},
        'correlation': {
            k: {key: val for key, val in v.items() if key != 'matrix'} 
            for k, v in correlation_results.items()
        } if 'correlation_results' in dir() else {},
        'multivariate': {
            'pca_explained_variance': multivariate_results.get('pca', {}).get('explained_variance_ratio', []).tolist() if 'multivariate_results' in dir() else [],
            'rf_importance': multivariate_results.get('rf_importance', {}).get('importance', {}).to_dict() if 'multivariate_results' in dir() else {}
        },
        'temporal': temporal_results if 'temporal_results' in dir() else {},
        'feature_selection': {
            'top_features': feature_selection_results.get('top_features', []) if 'feature_selection_results' in dir() else [],
            'importance_scores': feature_selector.importance_df.to_dict() if 'feature_selector' in dir() else {}
        }
    }
    
    # Save to JSON
    results_path = '../artifacts/reports/eda_results.json'
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"✅ Results saved to {results_path}")
    
    # Save feature list
    if 'feature_selector' in dir() and hasattr(feature_selector, 'top_features'):
        feature_list_path = '../artifacts/selected_features.txt'
        with open(feature_list_path, 'w') as f:
            for feature in feature_selector.top_features:
                f.write(f"{feature}\n")
        
        print(f"✅ Selected features saved to {feature_list_path}")
    
    return all_results

# Save results
analysis_results = save_analysis_results()

# %% [markdown]
# ### 10. Summary and Next Steps

# %%
print("\n" + "="*60)
print("NOTEBOOK 02 SUMMARY")
print("="*60)
print("""
✅ COMPLETED TASKS:
1. Univariate Analysis: Analyzed distributions of all features
   - Calculated skewness, kurtosis, normality tests
   - Identied outlier patterns

2. Bivariate Analysis: Feature vs Target relationships
   - Computed effect sizes (Cohen's d)
   - Calculated Information Values
   - Performed statistical significance tests

3. Correlation Analysis:
   - Pearson (linear), Spearman (monotonic), Kendall (ordinal)
   - Mutual Information (non-linear)
   - Identified multicollinearity patterns

4. Multivariate Analysis:
   - PCA for dimensionality reduction
   - t-SNE for visualization
   - Random Forest feature importance

5. Temporal Pattern Analysis:
   - Hourly and daily fraud patterns
   - Transaction velocity analysis
   - Time gap analysis

6. Feature Selection:
   - Ensemble importance scoring
   - Selected top features for modeling
   - Cumulative importance analysis

📊 KEY FINDINGS:
""")

# Print key findings
if 'feature_selection_results' in dir():
    top_features = feature_selection_results.get('top_features', [])[:10]
    print("\nTop 10 Most Important Features:")
    for i, feature in enumerate(top_features, 1):
        print(f"  {i}. {feature}")

if 'temporal_results' in dir() and 'hourly' in temporal_results:
    hourly_data = temporal_results['hourly']
    peak_hour = max(hourly_data, key=lambda x: x['fraud_count'])
    print(f"\nPeak fraud hour: {peak_hour['hour']:.0f}:00 ({peak_hour['fraud_count']} frauds)")

print("\n" + "-"*40)
print("NEXT STEPS")
print("-"*40)
print("""
Proceed to Notebook 03: Feature Engineering
- Create new features based on insights from EDA
- Engineer temporal features (rolling statistics, time gaps)
- Create interaction features
- Build feature pipeline
- Prepare data for modeling

Proceed to Notebook 04: Model Experimentation
- Train baseline models
- Experiment with different algorithms
- Handle class imbalance
- Cross-validation strategy
""")

print("\n" + "="*60)
print("END OF NOTEBOOK 02")
print("="*60)